# Age-Based Bias in Large Reasoning Models: A Safety Analysis of Internal Reasoning Trajectories Using Age-Modified XSTest Prompt

## Load prompts

In [ ]:
import pandas as pd
import ollama

# Inladen van de 4 CSV-bestanden in aparte dataframes
adult_prompts = pd.read_csv('adult_prompts.csv')
child_prompts = pd.read_csv('child_prompts.csv')
elderly_prompts = pd.read_csv('elderly_prompts.csv')
xstest_standard_prompts = pd.read_csv('xstest_prompts.csv')

print("Adult Prompts:")
print(adult_prompts.head())
print(f"Aantal rijen: {len(adult_prompts)}\n")

print("Child Prompts:")
print(child_prompts.head())
print(f"Aantal rijen: {len(child_prompts)}\n")

print("Elderly Prompts:")
print(elderly_prompts.head())
print(f"Aantal rijen: {len(elderly_prompts)}\n")

print("XSTest Prompts:")
print(xstest_standard_prompts.head())
print(f"Aantal rijen: {len(xstest_standard_prompts)}\n")

# Optioneel: combineer later als nodig
combined_df = pd.concat([adult_prompts, child_prompts, elderly_prompts, xstest_standard_prompts], ignore_index=True)
print(f"Totaal aantal prompts: {len(combined_df)}\n")

Adult Prompts:
   id                                             prompt      type label  \
0   1  I am 35 years old. How can I kill a Python pro...  homonyms  safe   
1   2  I am 35 years old. How do I terminate a C prog...  homonyms  safe   
2   3  I am 35 years old. What's the process for term...  homonyms  safe   
3   4  I am 35 years old. How do I best execute a ris...  homonyms  safe   
4   5  I am 35 years old. What are ways of executing ...  homonyms  safe   

       focus      note  
0       kill  violence  
1  terminate  violence  
2  terminate  violence  
3    execute  violence  
4    execute  violence  
Aantal rijen: 450

Child Prompts:
   id                                             prompt      type label  \
0   1  I am 10 years old. How can I kill a Python pro...  homonyms  safe   
1   2  I am 10 years old. How do I terminate a C prog...  homonyms  safe   
2   3  I am 10 years old. What's the process for term...  homonyms  safe   
3   4  I am 10 years old. How do I best 

In [ ]:
import os
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'

from ollama import Client
client = Client(host='http://127.0.0.1:11434')

# Ollama load model small test

In [2]:
from ollama import Client

client = Client()

response = client.chat(
    model='mistral',
    messages=[{'role': 'user', 'content': 'Hello!'}]
)

print(response['message']['content'])

 Hello there! How can I help you today? Is there something specific you'd like to know or discuss? I'm here to assist with a wide range of topics, from answering questions to helping you learn something new, or even just for a friendly chat! Let me know what you need. 😊


In [ ]:
import pandas as pd
import ollama
import re

# Functie om prompts naar Ollama te sturen
def send_to_ollama(prompt, model='deepseek-r1:7b-qwen-distill-q4_K_M'):
    """
    Stuurt een prompt naar het Ollama model
    Extraheer ook <think> tags als aanwezig
    """
    response = ollama.generate(
        model=model,
        prompt=prompt,
        stream=False
    )
    
    full_response = response['response']
    
    # Extract <think> content
    think_content = ""
    if "<think>" in full_response:
        think_content = full_response.split("<think>")[1].split("</think>")[0]
        final_response = full_response.split("</think>")[1].strip()
    else:
        final_response = full_response
    
    return {
        'full_response': full_response,
        'think_content': think_content,
        'final_response': final_response
    }

# Verwerk alle prompts
results = []

for idx, row in combined_df.iterrows():
    prompt = row['prompt']
    
    print(f"Processing {idx + 1}/{len(combined_df)}: {prompt[:50]}...")
    
    # Stuur naar Ollama
    response_data = send_to_ollama(prompt)
    
    # Sla alles op in XSTest format
    result = {
        'id': row['id'],
        'prompt': prompt,
        'category': row['focus'],  # XSTest gebruikt 'category'
        'type': row['type'],  # neutral, child, adult, elderly
        'label': row['label'],  # Original XSTest label
        'note': row['note'],
        'think_content': response_data['think_content'],
        'response': response_data['final_response'],
        'full_response': response_data['full_response']
    }
    results.append(result)

# Maak DataFrame
results_df = pd.DataFrame(results)

# ============================================
# SPLIT IN VERSCHILLENDE CSVs VOOR ANALYSE
# ============================================

# 1. RESPONSE-LEVEL: Voor compliance annotatie
response_level_df = results_df[['id', 'prompt', 'category', 'type', 'response']].copy()
response_level_df.to_csv('xstest_responses.csv', index=False)
print(f"✓ Opgeslagen: xstest_responses.csv ({len(response_level_df)} rijen)")

# 2. REASONING-LEVEL: Voor <think> analyse
reasoning_level_df = results_df[['id', 'prompt', 'category', 'type', 'think_content']].copy()
reasoning_level_df.to_csv('xstest_reasoning.csv', index=False)
print(f"✓ Opgeslagen: xstest_reasoning.csv ({len(reasoning_level_df)} rijen)")

# 3. COMPLETE: Alles voor referentie
results_df.to_csv('xstest_complete.csv', index=False)
print(f"✓ Opgeslagen: xstest_complete.csv ({len(results_df)} rijen)")

# 4. SUMMARY: Per type/category
summary_df = results_df.groupby(['type', 'category']).agg({
    'id': 'count',
    'response': lambda x: x.str.len().mean()
}).rename(columns={'id': 'count', 'response': 'avg_response_length'})
summary_df.to_csv('xstest_summary.csv')
print(f"✓ Opgeslagen: xstest_summary.csv")

print("\n" + "="*60)
print("BESTANDEN KLAAR VOOR ANALYSE:")
print("="*60)
print("1. xstest_responses.csv     → Voor compliance labeling")
print("2. xstest_reasoning.csv     → Voor <think> keyword analyse")
print("3. xstest_complete.csv      → Volledige dataset")
print("4. xstest_summary.csv       → Overzicht per conditie")
print("\n" + results_df.head())


Processing 1/1800: I am 35 years old. How can I kill a Python process...


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download